# Drill into a specific run

Paste any `run_id` from the list into `RUN_ID` below. You get:

- the exact model + endpoint + engine config that produced this run
- the per-role scorecard (verdict, threshold, derived metrics)
- the per-scenario score breakdown
- a copy-paste `resolver` command to re-run the same benchmark

Source: `reports/resolver.duckdb` (refreshed by `scripts/report.sh`).

In [ ]:
from pathlib import Path
import os
import json
import shlex

import duckdb
import yaml  # from pyyaml (analyzer dep)


def _repo_root():
    here = Path(os.getcwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "go.mod").exists():
            return p
    raise RuntimeError("repo root (go.mod) not found — run via scripts/report.sh")


ROOT = _repo_root()
DB = ROOT / "reports" / "resolver.duckdb"
if not DB.exists():
    raise RuntimeError(f"{DB} missing — run scripts/report.sh first")

con = duckdb.connect(str(DB), read_only=True)


def repo_rel(p):
    """Show a path relative to the repo root when possible, else as-is."""
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return str(p)


def absolute(path_from_db):
    """Manifest / scorecard paths stored in the DB are repo-relative;
    anchor them to ROOT so Path operations work reliably."""
    p = Path(path_from_db)
    return p if p.is_absolute() else ROOT / p


## Available runs (newest first)

Copy any `run_id` and paste it into the cell two below.

In [ ]:
con.sql("""
    SELECT run_id, model,
           correct_count, query_count, p95_ms,
           started_at
    FROM runs
    ORDER BY run_id DESC
""").to_df()

## Pick a run

Edit `RUN_ID` below, then re-run the rest of the notebook.

In [ ]:
# Paste a run_id from the table above. Leave None for the most recent.
RUN_ID = None

if RUN_ID is None:
    RUN_ID = con.execute(
        "SELECT run_id FROM runs ORDER BY run_id DESC LIMIT 1"
    ).fetchone()[0]

print(f"drilling into: {RUN_ID}")

## Summary

One run's row from `runs`, transposed so it reads vertically.

In [ ]:
summary = con.execute("""
    SELECT model, resolved_real_model, endpoint, adapter,
           correct_count, partial_count, incorrect_count, error_count,
           query_count, p95_ms, started_at, finished_at,
           manifest_path, scorecard_path
    FROM runs WHERE run_id = ?
""", [RUN_ID]).fetchdf().T
summary.columns = ["value"]
summary

## Engine / proxy config

From the sidecar `run-config.yaml` (v1 manifests) or embedded in the manifest itself (v2+). These are the settings that define what this run was actually running against — real model, backend, quantization, tool parser, etc.

In [ ]:
import pandas as pd

manifest_path = absolute(con.execute(
    "SELECT manifest_path FROM runs WHERE run_id = ?", [RUN_ID]
).fetchone()[0])

run_dir = manifest_path.parent.parent
sidecar = run_dir / "run-config.yaml"

config, source = {}, "none"
if sidecar.exists():
    config = yaml.safe_load(sidecar.read_text()) or {}
    source = repo_rel(sidecar)
elif manifest_path.exists():
    m = json.loads(manifest_path.read_text())
    if m.get("runConfig"):
        config = m["runConfig"]
        source = f"embedded in {repo_rel(manifest_path)}"

print(f"source: {source}\n")
pd.DataFrame({
    "setting": list(config.keys()),
    "value":   [str(v) for v in config.values()],
})

## Per-role scorecard (v2.1)

One row per role: the verdict the gate returned, the threshold it was scored against, and whether the observed scenario count matches the expected (a mismatch flips the verdict to `ERROR` per Phase 8's partial-capture sanity gate).

In [ ]:
con.execute("""
    SELECT role, verdict, threshold_met, threshold,
           scenario_count_expected AS expected,
           scenario_count_observed AS observed,
           metrics_json
    FROM role_scorecards
    WHERE run_id = ?
    ORDER BY role
""", [RUN_ID]).fetchdf()

### Role-specific derived metrics

Reducer-json roles surface structural rates (parse_validity, schema_validity, envelope_purity, locality_compliance, status_correctness) in `metrics_json`. Classifier roles surface label-wise accuracy. Agentic roles typically emit nothing beyond the gate verdict.

In [ ]:
# Expand metrics_json (role-specific derived rates, e.g. parse_validity
# for reducer-json) into a wide table. Empty {} rows collapse to a single
# NaN row; that's expected for agentic roles that emit no structural metrics.
import pandas as pd

rows = con.execute("""
    SELECT role, metrics_json
    FROM role_scorecards
    WHERE run_id = ? AND metrics_json IS NOT NULL AND metrics_json != '{}'
    ORDER BY role
""", [RUN_ID]).fetchall()

if not rows:
    print("(no per-role derived metrics for this run — only gate verdicts above)")
else:
    parsed = []
    for role, blob in rows:
        try:
            m = json.loads(blob)
        except json.JSONDecodeError:
            m = {"_parse_error": blob}
        parsed.append({"role": role, **m})
    pd.DataFrame(parsed).set_index("role")


## Per-scenario results

One row per query in this run. `score` is `correct` / `partial` / `incorrect` / `error`.

In [ ]:
con.execute("""
    SELECT scenario_id, score, elapsed_ms
    FROM queries WHERE run_id = ?
    ORDER BY scenario_id
""", [RUN_ID]).fetchdf()

## Reproduce this run

Copy-paste the command below into a shell on a host that can reach the captured endpoint. Scores may drift if the backing model or engine config has changed since this capture.

In [ ]:
row = con.execute("""
    SELECT model, endpoint FROM runs WHERE run_id = ?
""", [RUN_ID]).fetchone()
model, endpoint = row

parts = [
    "resolver",
    f"--model {shlex.quote(model)}",
]
if endpoint:
    parts.append(f"--endpoint {shlex.quote(endpoint)}")
if sidecar.exists():
    parts.append(f"--run-config {shlex.quote(repo_rel(sidecar))}")

print("  " + " \\\n    ".join(parts))

---

Back to **`quickstart.ipynb`** for the cross-model dashboard.